# Part 1 - Data Cleaning with Python and pandas
**Project: CareLoop Clinics - No-Show Guard**

CareLoop runs 8 clinics. About 1 in 5 booked patients does not turn up (a "no-show").
In this notebook we take the raw appointment file, check it, clean it, and save a tidy
version that Part 2 (SQL), Part 3 (machine learning) and Part 4 (Gemini) will use.

| Step | What we do |
|---|---|
| 1.1 | Read the CSV with plain Python (`csv.DictReader`) and reject rows that fail a type check |
| 1.2 | Load with pandas, remove duplicates, clean text, parse dates |
| 1.3 | Fix invalid ages, fill missing values, cap outliers |
| 1.4 | Create new columns and do NumPy calculations |
| 1.5 | Call the Open-Meteo API for each clinic |
| 1.6 | Export the clean files and the SQL load script |

## Setup
The first cell installs the libraries (needed on Google Colab; on a laptop it finishes quickly
when they are already installed). The second cell finds the project folder so that the same
relative paths (`data/...`, `outputs/...`) work on a laptop and on Colab.

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas numpy requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

# This notebook lives in notebooks/. The data lives one level up, in data/.
# BASE is the project's top folder, wherever the notebook is started from.
BASE = Path(".") if Path("data").exists() else Path("..")
DATA = BASE / "data"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)   # create outputs/ if it does not exist

print("Data files found:", sorted(p.name for p in DATA.glob("*.csv")))

Data files found: ['appointments.csv', 'clinics.csv', 'followup_notes.csv']


## Step 1.1 - Read the raw file with `csv.DictReader` and reject bad rows

**What:** read `appointments.csv` using only Python's built-in `csv` module (no pandas yet).
Every value in a CSV file is text. `parse_row` tries to turn the number columns into real
numbers. If a conversion fails, Python raises a `ValueError`; we catch it with `try/except`
and put that row in a "rejected" list instead of crashing.

**Why:** this is how a real data pipeline protects itself - one bad row must not stop the
whole load, and rejected rows must be saved so somebody can look at them later.

**Important idea:** a *type check* only asks "is this a number?". It does **not** ask
"is this number sensible?". An age of `150` or `-4` is a perfectly good integer, so it
passes here. We deal with those in Step 1.3 (a *validity check*).

In [3]:
import csv
import json
from collections import Counter

# Which columns must be numbers, and which Python type each one should become.
NUMERIC_COLUMNS = {
    "age": int,              # whole number of years
    "fee_inr": int,          # consultation fee in whole rupees
    "prior_no_shows": int,   # count of earlier no-shows
    "distance_km": float,    # can have decimals, e.g. 12.5
}


def parse_row(row):
    """Convert one raw CSV row (all text) into a row with proper number types.

    Raises ValueError if a number column is blank or is not a number.
    The error message starts with the column name so we know what went wrong.
    """
    clean = dict(row)  # copy, so the original row is left untouched
    for column, to_type in NUMERIC_COLUMNS.items():
        text = row[column].strip()
        try:
            clean[column] = to_type(text)      # int("") or float("abc") raises ValueError
        except ValueError as err:
            # Raise it again with the column name added, so the caller can record the reason.
            raise ValueError(f"{column}: {err}") from err
    return clean


accepted = []   # rows that passed the type check
rejected = []   # rows that failed, with the reason
rows_read = 0

# newline="" is what the csv module documentation recommends when opening CSV files.
with open(DATA / "appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)          # uses the header line as the dictionary keys
    print("Header:", reader.fieldnames)
    # start=2 because line 1 of the file is the header
    for line_number, row in enumerate(reader, start=2):
        rows_read += 1
        try:
            accepted.append(parse_row(row))
        except ValueError as err:
            rejected.append({
                "line_number": line_number,
                "appointment_id": row["appointment_id"],
                "reason": str(err),
                "row": row,                 # keep the raw row exactly as it was
            })

# Save the rejected rows so they can be reviewed (never throw bad data away silently).
with open(OUT / "rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

print("Rows read    :", rows_read)
print("Rows accepted:", len(accepted))
print("Rows rejected:", len(rejected))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read    : 1020
Rows accepted: 988
Rows rejected: 32


In [4]:
# Why were rows rejected? The reason text starts with the column name.
reasons = Counter(r["reason"].split(":")[0] for r in rejected)
print("Rejected by column:", dict(reasons))

print("\nExample rejected row:")
print(json.dumps(rejected[0], indent=2))

# A type check is not a validity check: impossible ages are still accepted here.
impossible = [r["age"] for r in accepted if r["age"] < 1 or r["age"] > 100]
print("\nImpossible ages that PASSED the type check:", impossible)
print("Types after parsing:", {c: type(accepted[0][c]).__name__ for c in NUMERIC_COLUMNS})

Rejected by column: {'distance_km': 20, 'age': 12}

Example rejected row:
{
  "line_number": 52,
  "appointment_id": "A0050",
  "reason": "distance_km: could not convert string to float: ''",
  "row": {
    "appointment_id": "A0050",
    "patient_id": "P117",
    "clinic_id": "C04",
    "booked_on": "2025-01-16",
    "appt_date": "2025-01-24",
    "age": "41",
    "gender": "M",
    "distance_km": "",
    "fee_inr": "600",
    "booking_channel": "Walk-in",
    "visit_type": "Consultation",
    "reminder_sent": "No",
    "prior_no_shows": "0",
    "status": "Attended"
  }
}

Impossible ages that PASSED the type check: [0, 150, -4, 250, 199]
Types after parsing: {'age': 'int', 'fee_inr': 'int', 'prior_no_shows': 'int', 'distance_km': 'float'}


**Reading the result**

- 1020 rows read, 988 accepted, 32 rejected. The 32 are 12 rows with a blank `age` and 20 rows
  with a blank `distance_km` (`int("")` and `float("")` both raise `ValueError`).
- The 5 impossible ages (0, 150, -4, 250, 199) were **accepted**, because each one is a valid
  integer. Catching them needs a business rule (age between 1 and 100), which comes in Step 1.3.
- Nothing is lost: the rejected rows sit in `outputs/rejected_rows.json`. In the pandas steps
  below we start again from the full raw file and *repair* these rows (fill the blanks)
  instead of dropping them.

## Step 1.2 - Inspect and clean, pass 1

**What:** load the same raw file with pandas and look at it (`shape`, `info()`, missing values,
duplicates). Then, in this order: (a) drop exact duplicate rows, (b) clean the messy
`booking_channel` spellings, (c) turn the two date columns into real dates, (d) drop rows
where the appointment date or the status is missing.

**Why:** duplicates would count the same appointment twice, messy spellings would split one
category into many, and we cannot use a row at all if we do not know *when* the appointment
was or *whether the patient came* (status is the thing we want to predict).

**Note:** this step starts again from the original `appointments.csv`. Step 1.1 was a separate
exercise with the `csv` module; its accepted/rejected lists are not used from here on.

In [5]:
import pandas as pd

# Read the raw file. pandas guesses the column types and turns blanks (and text like "N/A") into NaN.
df = pd.read_csv(DATA / "appointments.csv")

print("Shape (rows, columns):", df.shape)
print()
df.info()                      # column names, non-null counts and types
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())

Shape (rows, columns): (1020, 14)

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 170.6 KB

Missing values per column:
appointment_id      0
patient_id          0


`age` shows as `float64` even though ages are whole numbers. That is because the column has
blanks, and the usual integer type in pandas cannot hold a missing value (`NaN` is a float).

### (a) Drop exact duplicates
`duplicated()` marks a row as `True` when an earlier row is *exactly* the same in every column.

In [6]:
# Show one duplicated pair so we can see what a duplicate looks like.
example_id = df.loc[df.duplicated(), "appointment_id"].iloc[0]
display(df[df["appointment_id"] == example_id])

rows_raw = len(df)
df = df.drop_duplicates()          # keeps the first copy, removes the repeats
print(f"(a) after dropping duplicates: {rows_raw} -> {len(df)} rows")
print("appointment_id is now unique:", df["appointment_id"].is_unique)

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended


(a) after dropping duplicates: 1020 -> 1000 rows
appointment_id is now unique: True


### (b) Clean `booking_channel`
There are only 3 real channels (App, Phone, Walk-in) but people typed them in different ways:
extra spaces, different capital letters. To a computer `"App"`, `"app"` and `" App"` are three
different values. `.str.strip()` removes spaces at both ends and `.str.title()` makes every
word start with a capital letter (so `walk-in` becomes `Walk-In`).

In [7]:
print("Different raw spellings:", df["booking_channel"].nunique())
print(sorted(df["booking_channel"].unique()))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print("\nAfter cleaning:", df["booking_channel"].nunique(), "values")
print(df["booking_channel"].value_counts())
print(f"(b) after cleaning booking_channel: {len(df)} rows (no rows removed)")

Different raw spellings: 16
[' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']

After cleaning: 3 values
booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64
(b) after cleaning booking_channel: 1000 rows (no rows removed)


### (c) Convert the dates
`pd.to_datetime(..., errors="coerce")` turns text into real dates. `errors="coerce"` means:
if a value cannot be a date, do not crash - make it `NaT` ("Not a Time", the missing value for dates).

**Trap:** `isnull()` earlier found only 1 missing `appt_date`. Parsing finds more, because text
such as `TBD` or an impossible date such as `2025-02-30` is *not blank* but is still *not a date*.

In [8]:
# Keep the original text for a moment so we can show which values failed.
appt_text = df["appt_date"]

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")   # bad values become NaT
df["booked_on"] = pd.to_datetime(df["booked_on"])                    # this column is clean

print("appt_date values that are not valid dates:", df["appt_date"].isna().sum())
display(pd.DataFrame({
    "appointment_id": df.loc[df["appt_date"].isna(), "appointment_id"],
    "appt_date_as_typed": appt_text[df["appt_date"].isna()],
}))
print(f"(c) after converting dates: {len(df)} rows (no rows removed)")

appt_date values that are not valid dates: 6


,appointment_id,appt_date_as_typed
98,A0094,not booked
167,A0162,2025-06-31
303,A0296,2025-04-31
504,A0494,TBD
534,A0523,NaN
678,A0666,2025-02-30


(c) after converting dates: 1000 rows (no rows removed)


### (d) Drop rows with a missing date or missing status
We do **not** guess these. A made-up appointment date would give a wrong lead time, and a
made-up status would teach the model wrong answers.

In [9]:
rows_before = len(df)
print("Missing appt_date:", df["appt_date"].isna().sum(), "| missing status:", df["status"].isna().sum())

df = df.dropna(subset=["appt_date", "status"])
df = df.reset_index(drop=True)     # renumber the rows 0..n-1 after dropping

print(f"(d) after dropping missing date/status: {rows_before} -> {len(df)} rows")
print(f"\nRow count so far: {rows_raw} -> {rows_before} -> {len(df)}")
print("\nStatus counts:")
print(df["status"].value_counts())

Missing appt_date: 6 | missing status: 4
(d) after dropping missing date/status: 1000 -> 990 rows

Row count so far: 1020 -> 1000 -> 990

Status counts:
status
Attended    772
No-Show     218
Name: count, dtype: int64


**Reading the result**

- Raw shape is (1020, 14). Missing: `age` 12, `distance_km` 20, `status` 4, `appt_date` 1.
- 20 exact duplicates removed: 1020 -> 1000 rows. Every `appointment_id` is now unique.
- 16 raw spellings of `booking_channel` became 3 clean values: App, Phone, Walk-In.
- 6 appointment dates were unreadable (1 `N/A`, 2 pieces of text, 3 impossible calendar dates) and
  4 rows had no status. Dropping those 10 rows: 1000 -> 990.
- `age` and `distance_km` blanks are still there on purpose - we fill them in Step 1.3.